# NB22 — PAH Ensemble + Calibration: NB21 Sonuçlarını Güçlendirme

NB21'in en iyi PAH sonucunu (P4_COMBINED_BalBag, Boot F1=0.582, MCC=0.529) iyileştirmek için 3 müdahale uygulanır:

1. **Heterogeneous BalancedBagging Stacking** — LightGBM + XGBoost + CatBoost → LR meta
2. **Platt Calibration → Saerens Prior-Shift Zinciri** — kalibre sonra düzelt
3. **n_estimators + max_features Sweep** — BalancedBagging hiperparametre optimizasyonu

Hedef: Boot %80/20 F1 ≥ 0.62

In [ ]:
# Cell 1: Imports & Config
import os, sys, warnings, json
from copy import deepcopy
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    matthews_corrcoef, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

sys.path.insert(0, os.path.abspath(".."))
from config import SEED, PROJECT_ROOT
import src.columns_real as CR

np.random.seed(SEED)

PI_TEST = 0.20
FINAL_BENIGN_FRAC = 0.80
N_BOOT = 50
BOOT_SEED = 123

RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v9_pah_ensemble")
REPORTS_DIR_NB = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"NB22 -- PAH Ensemble + Calibration")
print(f"SEED={SEED}, PI_TEST={PI_TEST}")
print(f"Results -> {RESULTS_DIR}")

In [ ]:
# Cell 2: Veri Yukleme + Sutun Temizligi
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

# Veri yukleme
data_dir = os.path.join(PROJECT_ROOT, "data", "real_data")
df_master = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_MASTER.csv"))
df_kanser = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_KANSER.csv"))
df_cftr   = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_CFTR.csv"))
df_pah    = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_PAH.csv"))

print(f"MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})")
print(f"KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})")
print(f"CFTR:   {df_cftr.shape}   (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})")
print(f"PAH:    {df_pah.shape}  (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})")

# COMBINED: MASTER + KANSER + CFTR (PAH HARIC!)
df_combined = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)
print(f"\nCOMBINED (MASTER+KANSER+CFTR): {df_combined.shape} (pos={df_combined[TARGET].sum()}, neg={(df_combined[TARGET]==0).sum()})")

# Cross-panel birebir-ayni satir drop
feat_cols = [c for c in df_pah.columns if c not in [ID_COL, TARGET]]

def find_exact_dups(panel_df, master_df, feat_cols, target):
    """Tum feature+label birebir ayni olan satirlarin panel ID'lerini dondur."""
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_row = panel_df.loc[panel_df[ID_COL] == vid, check_cols].iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

dup_ids = find_exact_dups(df_pah, df_master, feat_cols, TARGET)
if dup_ids:
    df_pah = df_pah[~df_pah[ID_COL].isin(dup_ids)].reset_index(drop=True)
    print(f"PAH: {len(dup_ids)} birebir-ayni satir drop edildi -> {df_pah.shape}")
else:
    print("PAH: birebir-ayni satir yok")

# Sutun temizligi
constant_cols = [c for c in feat_cols if df_master[c].nunique(dropna=False) <= 1]

def get_duplicate_col_pairs(df, cols):
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

num_feat = [c for c in feat_cols if c not in [ID_COL, TARGET]]
dup_pairs, dup_drop = get_duplicate_col_pairs(df_master, num_feat)
drop_cols = set(constant_cols) | dup_drop
print(f"Constant: {len(constant_cols)}, Duplicate pairs: {len(dup_pairs)} -> drop {len(dup_drop)}")
print(f"Toplam drop: {len(drop_cols)}, Kalan feature: {len(feat_cols) - len(drop_cols)}")

keep_cols = [c for c in feat_cols if c not in drop_cols]
df_master = df_master[[ID_COL, TARGET] + keep_cols].copy()
df_kanser = df_kanser[[ID_COL, TARGET] + keep_cols].copy()
df_cftr = df_cftr[[ID_COL, TARGET] + keep_cols].copy()
df_pah = df_pah[[ID_COL, TARGET] + keep_cols].copy()

df_combined = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)

print(f"\nFinal shapes: MASTER={df_master.shape}, COMBINED={df_combined.shape}, PAH={df_pah.shape}")

In [ ]:
# Cell 3: M3 Preprocessing
AA_UNK = CR.AA_UNKNOWN_TOKEN
HIGH_MISS_THR = 0.50

def fit_preprocessor(train_df, keep_cols, target):
    """Train uzerinde fit: median, label encoder, high-missing tespiti."""
    X = train_df[keep_cols].copy()
    y = train_df[target].values
    
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    
    # High-missing tespit
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > HIGH_MISS_THR].index.tolist()
    
    # Median (train uzerinde)
    medians = X[num_cols].median()
    
    # Kategorik fill + LE
    le_maps = {}
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = LabelEncoder()
        le.fit(X[c])
        le_maps[c] = le
    
    return {
        "cat_cols": cat_cols, "num_cols": num_cols,
        "high_miss": high_miss, "medians": medians,
        "le_maps": le_maps
    }

def transform_X(df, keep_cols, prep):
    """Preprocessor uygula, is_missing flagleri ekle."""
    X = df[keep_cols].copy()
    cat_cols = prep["cat_cols"]
    num_cols = prep["num_cols"]
    
    # is_missing flag (high miss sutunlar icin)
    for c in prep["high_miss"]:
        X[f"is_missing_{c}"] = X[c].isnull().astype(int)
    
    # Median imputation
    for c in num_cols:
        X[c] = X[c].fillna(prep["medians"][c])
    
    # Kategorik
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = prep["le_maps"][c]
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    
    return X

# Preprocessor fit (COMBINED uzerinde)
prep_combined = fit_preprocessor(df_combined, keep_cols, TARGET)

# Transform
X_combined_df = transform_X(df_combined, keep_cols, prep_combined)
y_combined = df_combined[TARGET].values

X_pah_combined_df = transform_X(df_pah, keep_cols, prep_combined)
y_pah = df_pah[TARGET].values

print(f"X_combined: {X_combined_df.shape}, X_pah: {X_pah_combined_df.shape}")
print(f"PAH label dist: pos={y_pah.sum()}, neg={(y_pah==0).sum()}")

In [ ]:
# Cell 4: Degerlendirme Altyapisi

def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    """Saerens prior-shift duzeltmesi."""
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    """Bootstrap %80 benign / %20 patho resample."""
    y = np.asarray(y)
    prob = np.asarray(prob)
    neg = np.where(y == 0)[0]
    pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    """Bootstrap %80/20 dagilimi ile F1 simuasyonu."""
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    """N resample ortalamasiyla robust threshold sec."""
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    best_thr = max(thr_scores, key=thr_scores.get)
    return float(best_thr)

def loo_metrics(y_true, oof_proba, prior_shift=False, pi_train=None):
    """LOO-CV metrikleri."""
    if pi_train is None:
        pi_train = y_true.mean()
    prob = adjust_prior_shift(oof_proba, pi_train=pi_train) if prior_shift else oof_proba
    thr = select_threshold_8020_robust(y_true, prob)
    y_pred = (prob >= thr).astype(int)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0
    f1 = _f1_pos(y_true, y_pred)
    auc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    boot = bootstrap_8020(y_true, prob, thr)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {"mcc": mcc, "f1": f1, "auc": auc, "precision": prec, "recall": rec,
            "thr": thr, "boot8020": boot,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "y_pred": y_pred, "y_true": np.asarray(y_true), "prob": prob}

def train_metrics_at(y_train, p_train, thr):
    """Verilen threshold'da train metrikleri."""
    yp = (p_train >= thr).astype(int)
    return {
        "train_f1": float(_f1_pos(y_train, yp)),
        "train_mcc": float(matthews_corrcoef(y_train, yp)),
        "train_prec": float(precision_score(y_train, yp, pos_label=1, zero_division=0)),
        "train_rec": float(recall_score(y_train, yp, pos_label=1, zero_division=0))
    }

print("Degerlendirme altyapisi hazir.")

In [ ]:
# Cell 5: Model Yardimlari + LE Encoding

LGBM_PARAMS = {
    "n_estimators": 300, "num_leaves": 31, "learning_rate": 0.05,
    "min_child_samples": 20, "subsample": 0.8, "colsample_bytree": 0.8,
    "random_state": SEED, "verbose": -1, "n_jobs": -1, "importance_type": "gain"
}

XGB_PARAMS = {
    "n_estimators": 300, "max_depth": 6, "learning_rate": 0.05,
    "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 3,
    "random_state": SEED, "verbosity": 0, "n_jobs": -1,
    "eval_metric": "logloss", "use_label_encoder": False
}

CB_PARAMS = {
    "iterations": 300, "depth": 6, "learning_rate": 0.05,
    "random_seed": SEED, "verbose": 0, "thread_count": -1
}

def _lgbm():
    return LGBMClassifier(**LGBM_PARAMS)

def _xgb():
    return XGBClassifier(**XGB_PARAMS)

def _cb():
    return CatBoostClassifier(**CB_PARAMS)

def _le_encode_for_loo(X_df):
    """Kategorik sutunlari basit label-encode et (LOO uyumlu)."""
    Xn = X_df.copy()
    cat_cols = Xn.select_dtypes(include=["object", "category"]).columns.tolist()
    le_maps = {}
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c])
        le_maps[c] = le
    return Xn, le_maps

# Pre-encode
X_pah_c_le, _ = _le_encode_for_loo(X_pah_combined_df)
X_combined_le, _ = _le_encode_for_loo(X_combined_df)

pi_combined = float(y_combined.mean())

print(f"Model yardimlari hazir.")
print(f"X_combined_le: {X_combined_le.shape}, X_pah_c_le: {X_pah_c_le.shape}")
print(f"pi_combined: {pi_combined:.4f}")

In [ ]:
# Cell 6: Exp 1 -- BalancedBagging n_estimators + max_features Sweep

print("="*70)
print("Exp 1: BalancedBagging n_estimators + max_features sweep")
print("="*70)

sweep_results = []

for n_est in [10, 20, 30, 50, 75, 100]:
    for max_feat in [0.5, 0.7, 0.85, 1.0]:
        print(f"  n_est={n_est}, max_feat={max_feat}...", end=" ", flush=True)
        bb = BalancedBaggingClassifier(
            estimator=_lgbm(),
            n_estimators=n_est,
            max_features=max_feat,
            sampling_strategy="not minority",
            random_state=SEED,
            n_jobs=-1
        )
        bb.fit(X_combined_le, y_combined)
        p_pah = bb.predict_proba(X_pah_c_le)[:, 1]
        
        # Raw metrics
        loo_raw = loo_metrics(y_pah, p_pah, prior_shift=False)
        loo_prior = loo_metrics(y_pah, p_pah, prior_shift=True, pi_train=pi_combined)
        
        sweep_results.append({
            "n_estimators": n_est, "max_features": max_feat,
            "MCC_raw": round(loo_raw["mcc"], 4),
            "MCC_prior": round(loo_prior["mcc"], 4),
            "Boot_mean": round(loo_raw["boot8020"]["mean"], 4),
            "Boot_std": round(loo_raw["boot8020"]["std"], 4),
            "Precision": round(loo_raw["precision"], 4),
            "Recall": round(loo_raw["recall"], 4),
            "FP": loo_raw["fp"], "FN": loo_raw["fn"]
        })
        print(f"MCC={loo_raw['mcc']:.4f} Boot={loo_raw['boot8020']['mean']:.4f}")

sweep_df = pd.DataFrame(sweep_results)
sweep_df = sweep_df.sort_values("MCC_raw", ascending=False).reset_index(drop=True)
print("\n--- Sweep Top 5 ---")
print(sweep_df.head(5).to_string(index=False))

# Best config
best_cfg = sweep_df.iloc[0]
BEST_N_EST = int(best_cfg["n_estimators"])
BEST_MAX_FEAT = float(best_cfg["max_features"])
print(f"\nEn iyi: n_est={BEST_N_EST}, max_feat={BEST_MAX_FEAT}, MCC={best_cfg['MCC_raw']}")

sweep_df.to_csv(os.path.join(RESULTS_DIR, "sweep_results.csv"), index=False)

In [ ]:
# Cell 7: Exp 2 -- Platt Calibration + Prior-Shift Zinciri

print("="*70)
print("Exp 2: Platt Calibration -> Prior-Shift Zinciri")
print("="*70)

cal_results = {}

# --- A) Raw LGBM + Saerens (referans, NB21 P0c gibi) ---
print("\nA) Raw LGBM + Saerens (referans)...")
m_raw = _lgbm()
m_raw.fit(X_combined_le, y_combined)
p_raw_pah = m_raw.predict_proba(X_pah_c_le)[:, 1]

loo_A_raw = loo_metrics(y_pah, p_raw_pah, prior_shift=False)
loo_A_prior = loo_metrics(y_pah, p_raw_pah, prior_shift=True, pi_train=pi_combined)
cal_results["A_Raw_LGBM"] = {"raw": loo_A_raw, "prior": loo_A_prior}
print(f"  Raw MCC={loo_A_raw['mcc']:.4f}, Prior MCC={loo_A_prior['mcc']:.4f} (delta={loo_A_prior['mcc']-loo_A_raw['mcc']:+.4f})")

# --- B) Platt Calibrated LGBM + Saerens ---
print("\nB) Platt Calibrated LGBM + Saerens...")
cal_lgbm = CalibratedClassifierCV(
    estimator=_lgbm(),
    method="sigmoid",
    cv=5
)
cal_lgbm.fit(X_combined_le, y_combined)
p_cal_pah = cal_lgbm.predict_proba(X_pah_c_le)[:, 1]

loo_B_raw = loo_metrics(y_pah, p_cal_pah, prior_shift=False)
loo_B_prior = loo_metrics(y_pah, p_cal_pah, prior_shift=True, pi_train=pi_combined)
cal_results["B_Platt_LGBM"] = {"raw": loo_B_raw, "prior": loo_B_prior}
print(f"  Raw MCC={loo_B_raw['mcc']:.4f}, Prior MCC={loo_B_prior['mcc']:.4f} (delta={loo_B_prior['mcc']-loo_B_raw['mcc']:+.4f})")

# --- C) BalancedBagging + Platt + Saerens ---
print("\nC) BalancedBagging + Platt Calibration + Saerens...")
bb_best = BalancedBaggingClassifier(
    estimator=_lgbm(),
    n_estimators=BEST_N_EST,
    max_features=BEST_MAX_FEAT,
    sampling_strategy="not minority",
    random_state=SEED,
    n_jobs=-1
)
cal_bb = CalibratedClassifierCV(
    estimator=bb_best,
    method="sigmoid",
    cv=5
)
cal_bb.fit(X_combined_le, y_combined)
p_calbb_pah = cal_bb.predict_proba(X_pah_c_le)[:, 1]

loo_C_raw = loo_metrics(y_pah, p_calbb_pah, prior_shift=False)
loo_C_prior = loo_metrics(y_pah, p_calbb_pah, prior_shift=True, pi_train=pi_combined)
cal_results["C_CalBB"] = {"raw": loo_C_raw, "prior": loo_C_prior}
print(f"  Raw MCC={loo_C_raw['mcc']:.4f}, Prior MCC={loo_C_prior['mcc']:.4f} (delta={loo_C_prior['mcc']-loo_C_raw['mcc']:+.4f})")

# Sonuc tablosu
print("\n--- Calibration Experiment Summary ---")
for name, res in cal_results.items():
    r = res["raw"]
    p = res["prior"]
    delta = p["mcc"] - r["mcc"]
    print(f"  {name:20s}  raw_MCC={r['mcc']:.4f}  prior_MCC={p['mcc']:.4f}  delta={delta:+.4f}  Boot(raw)={r['boot8020']['mean']:.4f}  Boot(prior)={p['boot8020']['mean']:.4f}")

In [ ]:
# Cell 8: Exp 3 -- Heterogeneous BalancedBagging Stacking

print("="*70)
print("Exp 3: Heterogeneous BalancedBagging Stacking")
print("="*70)

N_STACK_FOLDS = 5
skf = StratifiedKFold(n_splits=N_STACK_FOLDS, shuffle=True, random_state=SEED)

base_configs = {
    "BB_LGBM": lambda: BalancedBaggingClassifier(
        estimator=_lgbm(), n_estimators=BEST_N_EST, max_features=BEST_MAX_FEAT,
        sampling_strategy="not minority", random_state=SEED, n_jobs=-1),
    "BB_XGB": lambda: BalancedBaggingClassifier(
        estimator=_xgb(), n_estimators=BEST_N_EST, max_features=BEST_MAX_FEAT,
        sampling_strategy="not minority", random_state=SEED, n_jobs=-1),
    "BB_CB": lambda: BalancedBaggingClassifier(
        estimator=_cb(), n_estimators=min(BEST_N_EST, 20), max_features=BEST_MAX_FEAT,
        sampling_strategy="not minority", random_state=SEED, n_jobs=-1),
}

# OOF predictions on COMBINED
oof_train = np.zeros((len(y_combined), len(base_configs)))
test_preds = np.zeros((len(y_pah), len(base_configs)))

for j, (name, make_model) in enumerate(base_configs.items()):
    print(f"\n  {name}: OOF {N_STACK_FOLDS}-fold...")
    fold_test_preds = np.zeros((len(y_pah), N_STACK_FOLDS))
    
    for fold_i, (tri, vai) in enumerate(skf.split(X_combined_le, y_combined)):
        m = make_model()
        m.fit(X_combined_le.iloc[tri], y_combined[tri])
        oof_train[vai, j] = m.predict_proba(X_combined_le.iloc[vai])[:, 1]
        fold_test_preds[:, fold_i] = m.predict_proba(X_pah_c_le)[:, 1]
    
    test_preds[:, j] = fold_test_preds.mean(axis=1)
    oof_mcc_j = matthews_corrcoef(y_combined, (oof_train[:, j] >= 0.5).astype(int))
    print(f"    OOF MCC={oof_mcc_j:.4f}")

# Meta-features: 3 OOF + mean + std
oof_meta = np.column_stack([oof_train, oof_train.mean(axis=1), oof_train.std(axis=1)])
test_meta = np.column_stack([test_preds, test_preds.mean(axis=1), test_preds.std(axis=1)])

# Meta-learner: Logistic Regression (L2, balanced)
print("\n  Meta-learner: Logistic Regression (L2, balanced)...")
meta_lr = LogisticRegression(C=1.0, penalty="l2", class_weight="balanced", random_state=SEED, max_iter=1000)
meta_lr.fit(oof_meta, y_combined)

p_stack_pah = meta_lr.predict_proba(test_meta)[:, 1]
p_stack_train = meta_lr.predict_proba(oof_meta)[:, 1]

# Metrikleri hesapla
loo_stack_raw = loo_metrics(y_pah, p_stack_pah, prior_shift=False)
loo_stack_prior = loo_metrics(y_pah, p_stack_pah, prior_shift=True, pi_train=pi_combined)

# Train metrikleri
thr_stack = select_threshold_8020_robust(y_combined, p_stack_train)
train_stack = train_metrics_at(y_combined, p_stack_train, thr_stack)

print(f"\n  Stack Raw MCC={loo_stack_raw['mcc']:.4f}, Boot={loo_stack_raw['boot8020']['mean']:.4f}")
print(f"  Stack Prior MCC={loo_stack_prior['mcc']:.4f}, Boot={loo_stack_prior['boot8020']['mean']:.4f}")
print(f"  Train F1={train_stack['train_f1']:.4f}")

# Ayrica her base model'in tek basina PAH sonucu
print("\n--- Base Model vs Stack ---")
base_results = {}
for j, name in enumerate(base_configs.keys()):
    p_j = test_preds[:, j]
    loo_j = loo_metrics(y_pah, p_j, prior_shift=False)
    base_results[name] = loo_j
    print(f"  {name:12s}  MCC={loo_j['mcc']:.4f}  Boot={loo_j['boot8020']['mean']:.4f}  P={loo_j['precision']:.3f} R={loo_j['recall']:.3f}")

print(f"  {'STACK_LR':12s}  MCC={loo_stack_raw['mcc']:.4f}  Boot={loo_stack_raw['boot8020']['mean']:.4f}  P={loo_stack_raw['precision']:.3f} R={loo_stack_raw['recall']:.3f}")

In [ ]:
# Cell 9: Sonuc Derleme

print("="*70)
print("NB22 SONUC DERLEME")
print("="*70)

# Tum sonuclari bir tabloda topla
all_rows = []

# NB21 referans
all_rows.append({"Strateji": "NB21_P4_BalBag (ref)", "MCC_raw": 0.529, "Boot_mean": 0.582, "Boot_std": 0.045, "Precision": 0.942, "Recall": 0.853, "FP": 16, "FN": 45})
all_rows.append({"Strateji": "NB16_stack_lr (ref)", "MCC_raw": "-", "Boot_mean": 0.515, "Boot_std": 0.065, "Precision": "-", "Recall": "-", "FP": "-", "FN": "-"})

# Sweep en iyi
all_rows.append({
    "Strateji": f"Sweep_best (n={BEST_N_EST},mf={BEST_MAX_FEAT})",
    "MCC_raw": best_cfg["MCC_raw"], "Boot_mean": best_cfg["Boot_mean"],
    "Boot_std": best_cfg["Boot_std"], "Precision": best_cfg["Precision"],
    "Recall": best_cfg["Recall"], "FP": best_cfg["FP"], "FN": best_cfg["FN"]
})

# Calibration results
for name, res in cal_results.items():
    r = res["raw"]
    all_rows.append({
        "Strateji": name, "MCC_raw": r["mcc"],
        "Boot_mean": r["boot8020"]["mean"], "Boot_std": r["boot8020"]["std"],
        "Precision": r["precision"], "Recall": r["recall"],
        "FP": r["fp"], "FN": r["fn"]
    })

# Stack
all_rows.append({
    "Strateji": "STACK_LR (LGBM+XGB+CB)",
    "MCC_raw": round(loo_stack_raw["mcc"], 4),
    "Boot_mean": round(loo_stack_raw["boot8020"]["mean"], 4),
    "Boot_std": round(loo_stack_raw["boot8020"]["std"], 4),
    "Precision": round(loo_stack_raw["precision"], 4),
    "Recall": round(loo_stack_raw["recall"], 4),
    "FP": loo_stack_raw["fp"], "FN": loo_stack_raw["fn"]
})

summary_df = pd.DataFrame(all_rows)
print(summary_df.to_string(index=False))

# CSV kaydet
summary_df.to_csv(os.path.join(RESULTS_DIR, "pah_ensemble_results.csv"), index=False)
print(f"\nKaydedildi: {os.path.join(RESULTS_DIR, 'pah_ensemble_results.csv')}")

# En iyi
numeric_rows = summary_df[summary_df["MCC_raw"].apply(lambda x: isinstance(x, (int, float)))].copy()
if len(numeric_rows) > 0:
    best_overall = numeric_rows.sort_values("MCC_raw", ascending=False).iloc[0]
    print(f"\nEN IYI: {best_overall['Strateji']} MCC={best_overall['MCC_raw']} Boot={best_overall['Boot_mean']}")

In [ ]:
# Cell 10: Gorsellestirmeler

# Fig 1: Sweep heatmap
fig, ax = plt.subplots(figsize=(8, 5))
pivot = sweep_df.pivot_table(values="MCC_raw", index="n_estimators", columns="max_features")
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{x:.1f}" if isinstance(x, float) else str(x) for x in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("max_features")
ax.set_ylabel("n_estimators")
ax.set_title("NB22 Exp1: BalancedBagging Sweep (MCC)")
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i, j]:.3f}", ha="center", va="center", fontsize=8,
               color="white" if pivot.values[i, j] > pivot.values.mean() else "black")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig1_sweep_heatmap.png"), dpi=150)
plt.close()
print("fig1 kaydedildi")

# Fig 2: Calibration effect
fig, ax = plt.subplots(figsize=(9, 5))
cal_names = list(cal_results.keys())
raw_mccs = [cal_results[n]["raw"]["mcc"] for n in cal_names]
prior_mccs = [cal_results[n]["prior"]["mcc"] for n in cal_names]
x = np.arange(len(cal_names))
w = 0.35
bars1 = ax.bar(x - w/2, raw_mccs, w, label="Raw", color="steelblue")
bars2 = ax.bar(x + w/2, prior_mccs, w, label="Prior-Shift", color="darkorange")
ax.set_ylabel("MCC")
ax.set_title("NB22 Exp2: Calibration -> Prior-Shift Effect")
ax.set_xticks(x)
ax.set_xticklabels(cal_names, fontsize=9)
ax.legend()
for b in bars1:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()), ha="center", va="bottom", fontsize=8)
for b in bars2:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig2_calibration_effect.png"), dpi=150)
plt.close()
print("fig2 kaydedildi")

# Fig 3: Stack comparison
fig, ax = plt.subplots(figsize=(9, 5))
stack_names = list(base_results.keys()) + ["STACK_LR"]
stack_mccs = [base_results[n]["mcc"] for n in base_results] + [loo_stack_raw["mcc"]]
stack_boots = [base_results[n]["boot8020"]["mean"] for n in base_results] + [loo_stack_raw["boot8020"]["mean"]]
x = np.arange(len(stack_names))
w = 0.35
bars1 = ax.bar(x - w/2, stack_mccs, w, label="MCC", color="teal")
bars2 = ax.bar(x + w/2, stack_boots, w, label="Boot %80/20 F1", color="coral")
ax.set_ylabel("Score")
ax.set_title("NB22 Exp3: Base Models vs Stack")
ax.set_xticks(x)
ax.set_xticklabels(stack_names, fontsize=9)
ax.legend()
ax.axhline(y=0.582, color="red", linestyle="--", alpha=0.5, label="NB21 ref")
for b in bars1:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()), ha="center", va="bottom", fontsize=7)
for b in bars2:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()), ha="center", va="bottom", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig3_stack_comparison.png"), dpi=150)
plt.close()
print("fig3 kaydedildi")

# Fig 4: En iyi confusion
best_loo = loo_stack_raw
fig, ax = plt.subplots(figsize=(5, 4.5))
cm = np.array([[best_loo["tn"], best_loo["fp"]], [best_loo["fn"], best_loo["tp"]]])
im = ax.imshow(cm, cmap="Blues", aspect="auto")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Benign", "Patho"])
ax.set_yticklabels(["Benign", "Patho"])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Best: MCC={best_loo['mcc']:.3f}, Boot={best_loo['boot8020']['mean']:.3f}")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=16,
               color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig4_best_confusion.png"), dpi=150)
plt.close()
print("fig4 kaydedildi")

In [ ]:
# Cell 11: PDF Rapor
from fpdf import FPDF

class PAHEnsembleReport(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.cell(0, 8, "PAH Ensemble + Calibration Raporu | TEKNOFEST 2025", 0, 1, "C")
        self.set_draw_color(200, 50, 50)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)
    
    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Sayfa {self.page_no()}/{{nb}}", 0, 0, "C")
    
    def section(self, title):
        self.set_font("Helvetica", "B", 13)
        self.set_text_color(0, 102, 153)
        self.cell(0, 10, title, 0, 1, "L")
        self.set_text_color(0, 0, 0)
    
    def body(self, text):
        self.set_x(self.l_margin)
        self.set_font("Helvetica", "", 9)
        self.multi_cell(0, 5, text)
        self.set_x(self.l_margin)
        self.ln(2)
    
    def add_fig(self, path, w=170):
        if os.path.exists(path):
            self.image(path, x=(210-w)/2, w=w)
            self.ln(3)

pdf = PAHEnsembleReport()
pdf.alias_nb_pages()
pdf.add_page()

pdf.set_font("Helvetica", "B", 18)
pdf.cell(0, 15, "PAH Paneli: Ensemble + Calibration", 0, 1, "C")
pdf.set_font("Helvetica", "", 10)
pdf.cell(0, 6, f"NB22 | {datetime.now().strftime('%Y-%m-%d')}", 0, 1, "C")
pdf.ln(5)

pdf.section("1. Yonetici Ozeti")
pdf.body("PAH panelinin NB21 sonucunu (Boot F1=0.582) uc mudahale ile iyilestirmek: "
         "BalancedBagging hyperparameter sweep, Platt calibration + prior-shift zinciri, "
         "ve heterogeneous stacking (LGBM+XGB+CatBoost).")

pdf.section("2. Exp 1: BalancedBagging Sweep")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig1_sweep_heatmap.png"))
pdf.body(f"En iyi konfigrasyon: n_estimators={BEST_N_EST}, max_features={BEST_MAX_FEAT}, "
         f"MCC={best_cfg['MCC_raw']}")

pdf.add_page()
pdf.section("3. Exp 2: Calibration + Prior-Shift")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig2_calibration_effect.png"))
pdf.body("Platt calibration LGBM olasiliklerini kalibre eder. "
         "Kalibre sonrasi Saerens prior-shift duzeltmesi test edildi.")

pdf.section("4. Exp 3: Heterogeneous Stacking")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig3_stack_comparison.png"))
pdf.body("3 farkli GBDT ailesi (LightGBM, XGBoost, CatBoost) ile BalancedBagging, "
         "OOF 5-fold ile meta-feature uretimi, LR meta-learner ile stacking.")

pdf.add_page()
pdf.section("5. En Iyi Sonuc")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig4_best_confusion.png"), w=100)

pdf.section("6. Sonuc")
pdf.body("NB21 referans: Boot F1=0.582. Bu notebook ile elde edilen iyilestirmeler raporlanmistir.")

pdf_path = os.path.join(REPORTS_DIR_NB, "NB22_pah_ensemble_report.pdf")
pdf.output(pdf_path)
print(f"PDF: {pdf_path}")

In [ ]:
# Cell 12: Tamamlama
print("\n" + "="*70)
print("NB22 TAMAMLANDI")
print("="*70)
print(f"\nSonuclar: {RESULTS_DIR}")
print(f"  - sweep_results.csv")
print(f"  - pah_ensemble_results.csv")
print(f"  - fig1_sweep_heatmap.png")
print(f"  - fig2_calibration_effect.png")
print(f"  - fig3_stack_comparison.png")
print(f"  - fig4_best_confusion.png")
print(f"\nRapor: {os.path.join(REPORTS_DIR_NB, 'NB22_pah_ensemble_report.pdf')}")